# Baseline evidence analysis

This output-free notebook is a thin caller of `minires.evaluate_records`. It does not preprocess, split, fit, or calculate metrics independently. Execute only with private local paths, save the executed copy beneath `private/`, and inspect only the aggregate allowlisted summaries here.

In [ ]:
from pathlib import Path

from minires import (
    EvaluationConfig,
    LearnedBaseline,
    LegacyProvenance,
    PhysicalBaseline,
    evaluate_records,
    load_legacy_reference,
)

In [ ]:
records_path = Path("private/evaluation-records.json")
comparison_paths = [
    Path("data/3d_print_miniatures_base.csv"),
    Path("data/3d_print_miniatures_data.csv"),
]
legacy_artifacts_path = Path("private/legacy-artifacts")
split_manifest_path = Path("private/baseline-evidence/split.json")
run_root = Path("private/baseline-evidence/notebook-runs")
config = EvaluationConfig(
    resin_density_g_per_ml=None,  # Set only from attested evidence.
    volume_unit="mm3",
    scope_confirmed=None,  # Set True only from attested evidence.
    seed=0,
)

In [ ]:
baseline_factories = {
    # Initialize the learned runtime first so TensorFlow's deterministic
    # dataset mode is configured before loading the legacy model.
    "clean_fixed_configuration": LearnedBaseline,
    "physical": PhysicalBaseline,
    "legacy_reference": lambda: load_legacy_reference(
        legacy_artifacts_path, provenance=LegacyProvenance.unknown()
    ),
}
results = {}
for name, factory in baseline_factories.items():
    results[name] = evaluate_records(
        records_path,
        config,
        factory(),
        reconcile_with=comparison_paths if name == "physical" else [],
        output_dir=run_root / name,
        split_manifest=split_manifest_path.with_name(
            f"{split_manifest_path.stem}-{name}.json"
        ),
    )
public_summaries = {name: result.to_dict(public=True) for name, result in results.items()}

In [ ]:
{
    name: {
        "status": summary["status"],
        "blockers": summary["blockers"],
        "data_quality": summary["data_quality"],
        "metrics": summary["metrics"],
        "grouped_evaluation": summary.get("grouped_evaluation"),
    }
    for name, summary in public_summaries.items()
}